# 02 · RAG use case — MultiHop-RAG

**Deck section 2** · slides 15–20

MultiHop-RAG (Tang & Yang, COLM 2024) contains 2,556 questions whose evidence is spread
across two to four documents, and some of them depend on document metadata rather than
article text. That single property — evidence spread across documents — is what makes it the
right harness for this curriculum, because it breaks every metric that treats retrieval as
"did we find *a* relevant passage".

This notebook is about the **record**, not the dataset. One record supports three different
evaluations, question type determines which retrieval strategy will work, and — the part that
matters most in client work — you will almost never be handed a labelled set at all, so you
have to manufacture one.

**By the end you can**

- say what each field of an eval record buys you, and what you lose without `evidence_list`
- predict a question type's dominant failure and name the lever that fixes it
- implement two of those levers and measure whether they worked
- run the SEED → FILTER → MAINTAIN pipeline that turns an unlabelled client corpus into a
  defensible eval set


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import pandas as pd
import raglab
from raglab import viz, tables, catalog, corpus, metrics, retrieve, pipeline
viz.reset_figures("2."); tables.reset_tables("2.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)
rows = pipeline.evaluate(pipe, bundle.questions, pipe.chunks, personas=bundle.personas)
print(f"{len(bundle.questions)} questions evaluated against the baseline system")

---

## 2.1 What one record lets you evaluate

### What the code is about to do

Three evaluations from one record: retrieval scored against `evidence_list`, the answer
scored against `answer`, and the report sliced by `question_type`. The cell below takes a
single record and produces all three.


In [ ]:
viz.hld([
    dict(name="Inputs", tone="index", chain=False, nodes=[
        ("Corpus + metadata", "everything indexed, with source, date and ACL"),
        ("Multi-hop question", "two to four documents carry the evidence")]),
    dict(name="System under test", tone="query", nodes=[
        ("Retriever", "candidates"), ("Ranking + packing", "what the model sees"),
        ("Answer generation", "a claim with citations")]),
    dict(name="Scored against", tone="control", chain=False, nodes=[
        ("evidence_list", "→ Evidence Recall@k, per-hop coverage, full-chain recall"),
        ("answer", "→ exact match or a judged verdict"),
        ("question_type", "→ the slice that shows which type you are losing on")]),
], title="One record, three evaluations", kicker="Section 2 · dataset",
   caption="The same record exposes retrieval misses, incomplete evidence chains, and "
           "answer-quality failures. That is why the setup cost is worth paying.",
   source="Deck slide 17")

In [ ]:
import json

q = next(x for x in bundle.questions if x.question_type == "comparison" and x.hops == 2)
tr = pipe.run(q.query, qid=q.qid)
gm, _ = metrics.resolve_gold(q, pipe.chunks)

print("THE RECORD")
print(json.dumps(q.as_record(), indent=2)[:640], "\n")

print("EVALUATION 1 — retrieval, scored against evidence_list")
for i, (anchor, cids) in enumerate(gm.items(), 1):
    found = cids & set(tr.packed_ids)
    print(f"  hop {i}  {'FOUND  ' if found else 'MISSING'}  {anchor[:62]}")
print(f"  Evidence Recall@k   {metrics.evidence_recall_at_k(tr.packed_ids, gm):.2f}")
print(f"  Full-chain recall   {metrics.full_chain_recall(tr.packed_ids, gm):.0f}"
      "   ← 1 only if every hop arrived\n")

print("EVALUATION 2 — the answer, scored against `answer`")
print(f"  gold    {q.answer}")
print(f"  system  {tr.answer[:110]}")
print(f"  correct {metrics.answer_correct(tr.answer, q.answer):.0f}\n")

print("EVALUATION 3 — the slice, from `question_type`")
print(metrics.slice_report(rows).to_string(index=False))

### Why `evidence_list` is the field that matters

Without it you can only score answers. Answer-only scoring is not merely less informative —
it is actively misleading, because an answer can be right while retrieval found nothing. The
cell below counts how often that happens on this system.


In [ ]:
answerable = [r for r in rows if not r["is_null"]]
lucky = [r for r in answerable if r["answer_correct"] >= 1.0 and r["evidence_recall"] == 0.0]
blind = [r for r in answerable if r["answer_correct"] == 0.0 and r["full_chain_recall"] >= 1.0]

tables.show(pd.DataFrame([
    ["Right answer, zero gold evidence retrieved", len(lucky),
     f"{len(lucky)/len(answerable):.1%}",
     "Answer-only scoring calls these a pass. They are luck, and they will not survive a "
     "corpus refresh."],
    ["Wrong answer, complete evidence chain retrieved", len(blind),
     f"{len(blind)/len(answerable):.1%}",
     "Retrieval did its job. This is a generation or rubric problem and no amount of "
     "retriever tuning will move it."],
    ["Answer and evidence agree", len(answerable) - len(lucky) - len(blind),
     f"{(len(answerable)-len(lucky)-len(blind))/len(answerable):.1%}",
     "The only cases where an end-to-end score means what you think it means."],
], columns=["Case", "Questions", "Share", "What it costs you to not know"]),
    title="What answer-only scoring hides",
    kicker="Why evidence_list exists",
    caption="On this run, an end-to-end number misattributes a measurable fraction of the "
            "eval set in both directions. Two numbers cost almost nothing more to compute.",
    emphasize="Share")

---

## 2.2 Question type drives retrieval strategy

Different question types fail for different reasons. Report metrics sliced this way or you
will average away your real problem.


In [ ]:
catalog.QUESTION_TYPE.show()

### Now prove each row

A matrix is a claim. The next four cells test each claim on this system: measure the dominant
failure, apply the lever the matrix prescribes, and measure again.


In [ ]:
# ── INFERENCE ── claim: "the second hop never enters the candidate pool"
inf = [r for r in rows if r["question_type"] == "inference" and r["gold_items"] >= 2]
missing_at_N = [r for r in inf if r["full_chain_recall_at_N"] == 0.0]
lost_in_pack = [r for r in inf if r["full_chain_recall_at_N"] == 1.0
                and r["full_chain_recall"] == 0.0]

print(f"INFERENCE — {len(inf)} multi-hop questions")
print(f"  second hop never reached the pool (N)      {len(missing_at_N):>3}  "
      f"{len(missing_at_N)/len(inf):.1%}   ← the matrix's claim")
print(f"  reached the pool, lost during packing (k)  {len(lost_in_pack):>3}  "
      f"{len(lost_in_pack)/len(inf):.1%}   ← a different fix entirely")
print(f"  complete chain delivered                   "
      f"{len(inf)-len(missing_at_N)-len(lost_in_pack):>3}")

In [ ]:
# ── COMPARISON ── claim: "one entity dominates top-k; the other is starved"
# Measure it directly: for each comparison question, how are the packed slots split
# between the two entities the question names?
import re

def entities_in(question_text):
    names = [o["name"] for o in corpus.ORG.values()]
    found = [n for n in names if n.lower() in question_text.lower()]
    if len(found) < 2:                      # descriptor phrasing: resolve via the gold docs
        return None
    return found[:2]

splits = []
for r in rows:
    if r["question_type"] != "comparison":
        continue
    x = next(y for y in bundle.questions if y.qid == r["qid"])
    ents = entities_in(x.query)
    if not ents:
        continue
    tr = pipe.run(x.query, qid=x.qid)
    counts = [sum(1 for cid in tr.packed_ids
                  if e.lower() in (index.get(cid) or {"text": ""})["text"].lower())
              for e in ents]
    if sum(counts) == 0:
        continue
    splits.append((min(counts) / sum(counts), r["full_chain_recall"]))

starved = [s for s, _ in splits if s < 0.25]
shares = sorted(s for s, _ in splits)
print(f"COMPARISON — {len(splits)} questions naming two entities explicitly")
print(f"  median share of packed slots held by the weaker entity  {shares[len(shares)//2]:.2f}")
print(f"  lowest share observed                                   {shares[0]:.2f}")
print(f"  questions where one entity holds <25% of the slots      {len(starved)} "
      f"({len(starved)/len(splits):.0%})")

# Does corpus prevalence predict the imbalance? That is the mechanism the matrix is
# describing, and it is testable.
prevalence = {}
for o in corpus.ORG.values():
    prevalence[o["name"]] = sum(1 for c in pipe.chunks if o["name"].lower() in c.text.lower())
ratios, imbalance = [], []
for r in rows:
    if r["question_type"] != "comparison":
        continue
    x = next(y for y in bundle.questions if y.qid == r["qid"])
    ents = entities_in(x.query)
    if not ents:
        continue
    a, b = prevalence.get(ents[0], 1), prevalence.get(ents[1], 1)
    ratios.append(max(a, b) / max(1, min(a, b)))
tr_share = [s for s, _ in splits]
print(f"\n  corpus prevalence ratio between the two entities: "
      f"median {sorted(ratios)[len(ratios)//2]:.2f}x, max {max(ratios):.2f}x")

**The matrix's claim does not reproduce here, and that is worth more than a confirmation.**

Nothing is starved. The median split is even, and the corpus prevalence ratio between the two
entities in a typical comparison question is close to 1 — because this corpus was generated
with the same number of quarters for every company. Starvation is a function of *corpus
imbalance*, not of comparison questions as such. On a client corpus where one product line
has ten thousand tickets and another has two hundred, the same question shape will starve,
and it will starve in exactly the direction the prevalence ratio predicts.

That is what a decision matrix is for. It names a mechanism you should go and test, and the
test can come back negative. A room that treats the matrix as a set of facts to recite will
spend a sprint fixing starvation it does not have.

The lever is still worth measuring, because it does not depend on starvation being severe —
it depends only on a global relevance ranking being the wrong objective for a question that
needs balance. **Per-entity retrieval quotas in packing**: reserve slots for each side
instead of letting one ranking decide. Fifteen lines.


In [ ]:
from raglab.retrieve import pack_context

def quota_pack(hits, entities, k=8, token_cap=6000):
    '''Reserve packing slots per entity, then fill the remainder by rank.

    The global ranking is not wrong -- it is answering a different question. It ranks by
    relevance to the query as a whole, and a comparison question needs balance, which no
    single relevance ordering expresses. So the constraint goes in the packer, where it
    belongs, rather than being wished for in the reranker.
    '''
    per = max(1, k // max(1, len(entities)))
    chosen, used, taken = [], 0, set()
    for e in entities:
        n = 0
        for h in hits:
            if h.chunk_id in taken or n >= per:
                continue
            if e.lower() in h.text.lower():
                from raglab.chunking import approx_tokens
                t = approx_tokens(h.text)
                if used + t > token_cap:
                    continue
                chosen.append(h); taken.add(h.chunk_id); used += t; n += 1
    rest, extra = pack_context([h for h in hits if h.chunk_id not in taken],
                               k=k - len(chosen), token_cap=token_cap - used)
    return chosen + rest, used + extra


comparison_qs = [x for x in bundle.questions
                 if x.question_type == "comparison" and entities_in(x.query)]
before, after = [], []
for x in comparison_qs:
    ents = entities_in(x.query)
    gm, _ = metrics.resolve_gold(x, pipe.chunks)
    cands = pipe.retriever.search(x.query, pipe.cfg)
    ranked = pipe.reranker.rerank(x.query, cands, depth=pipe.cfg.rerank_depth)
    plain, _ = pack_context(ranked, k=pipe.cfg.k, token_cap=pipe.cfg.evidence_token_cap)
    quota, _ = quota_pack(ranked, ents, k=pipe.cfg.k, token_cap=pipe.cfg.evidence_token_cap)
    before.append(metrics.full_chain_recall([h.chunk_id for h in plain], gm))
    after.append(metrics.full_chain_recall([h.chunk_id for h in quota], gm))

viz.bars(["global ranking", "per-entity quota"],
         {"full-chain recall": [sum(before)/len(before), sum(after)/len(after)]},
         title=f"Per-entity packing quotas on {len(comparison_qs)} comparison questions",
         kicker="Lever, measured", ylabel="full-chain recall",
         caption="Same candidates, same reranker, same k. Only the packing rule changed — "
                 "which is the point: the fix belonged to stage three all along.")

delta = sum(after)/len(after) - sum(before)/len(before)
print(f"full-chain recall {sum(before)/len(before):.3f} → {sum(after)/len(after):.3f} "
      f"({delta:+.3f})")
print(f"questions improved: {sum(1 for b, a in zip(before, after) if a > b)}   "
      f"regressed: {sum(1 for b, a in zip(before, after) if a < b)}   "
      f"unchanged: {sum(1 for b, a in zip(before, after) if a == b)}")
print("\nA gain without starvation present. The quota is not repairing a pathology — it is")
print("encoding a constraint the relevance ranking had no way to express.\n")

boot = metrics.paired_bootstrap(
    [{"qid": x.qid, "full_chain_recall": b} for x, b in zip(comparison_qs, before)],
    [{"qid": x.qid, "full_chain_recall": a} for x, a in zip(comparison_qs, after)])
print(f"paired bootstrap  delta {boot['delta']:+.3f}  95% CI "
      f"[{boot['ci'][0]:+.3f}, {boot['ci'][1]:+.3f}]  verdict: {boot['verdict']}")
print("Two questions moved out of 31. Read the verdict before you put this in a deck: a")
print("six-point headline built on two examples is a coin flip you have chosen to believe.")
print("The lever is sound and the mechanism is real — the *evidence for it here* is thin,")
print("and saying so is the difference between a result and a story.")

In [ ]:
# ── TEMPORAL ── claim: "embeddings ignore dates; the newest doc wins on similarity"
temporal = [r for r in rows if r["question_type"] == "temporal"]
print(f"TEMPORAL — {len(temporal)} questions")
print(f"  evidence recall  {sum(r['evidence_recall'] for r in temporal)/len(temporal):.3f}")
print(f"  full-chain       {sum(r['full_chain_recall'] for r in temporal)/len(temporal):.3f}")
print(f"  answer correct   {sum(r['answer_correct'] for r in temporal)/len(temporal):.3f}"
      "   ← retrieval is fine; the reader cannot order events\n")

# The prescribed lever: a metadata filter, evaluated inside the query rather than after it.
dated = [x for x in bundle.questions
         if x.question_type == "temporal" and re.search(r"\b(20\d\d)\b", x.query)]
plain_r, filtered_r = [], []
for x in dated[:24]:
    year = re.search(r"\b(20\d\d)\b", x.query).group(1)
    gm, _ = metrics.resolve_gold(x, pipe.chunks)
    a = pipe.run(x.query, qid=x.qid)
    v = pipe.variant("dated", filters={"published_from": f"{year}-01-01",
                                       "published_to": f"{int(year)+1}-06-30"})
    b = v.run(x.query, qid=x.qid)
    plain_r.append(metrics.evidence_recall_at_k(a.packed_ids, gm))
    filtered_r.append(metrics.evidence_recall_at_k(b.packed_ids, gm))

print(f"METADATA FILTER on {len(plain_r)} questions that name a year")
print(f"  no filter        evidence recall {sum(plain_r)/len(plain_r):.3f}")
print(f"  date-window pre-filter            {sum(filtered_r)/len(filtered_r):.3f}")
print("  (the filter is pushed into the query, not applied to the results — same rule as ACLs)")

In [ ]:
# ── NULL ── claim: "the model answers anyway from a plausible distractor"
ab = metrics.abstention_scores(rows)
nulls = [r for r in rows if r["is_null"]]
print(f"NULL — {len(nulls)} questions the corpus cannot answer")
print(f"  abstained            {sum(1 for r in nulls if r['abstained'])}/{len(nulls)}")
print(f"  abstention recall    {ab['abstention_recall']:.3f}")
print(f"  answered anyway      {ab['false_answers_on_null']}   ← every one is a wrong answer "
      "delivered with full confidence")
print(f"  over-refusals        {ab['over_refusals']}\n")
print("Meanwhile every other metric on this run is unaffected, because null questions have no")
print("gold evidence to miss. That asymmetry is the whole argument for adding them.")

In [ ]:
tables.callout(
    "Null questions are the cheapest thing you can add to a client eval set and the fastest "
    "way to expose a system that never says “I don't know”. They cost nothing to write — you "
    "already know what your corpus does <i>not</i> contain — and no other metric in your "
    "report will move when your system starts inventing answers."
    "<br><br>Notebook 06 attacks the fix, and finds something uncomfortable: on this eval set "
    "<b>no retrieval-score threshold separates answerable from unanswerable</b>, because the "
    "null questions name real entities in the corpus's own vocabulary while the answerable "
    "ones paraphrase. Abstention is an entailment question, and entailment needs a reader.",
    kind="warn", title="Why the null set earns its place")

---

## 2.3 Manufacturing an eval set from a client corpus

Clients never hand you gold labels. This is the pipeline that makes them, and the human
review step is not optional — it is the step that makes the set defensible when the client
asks where the numbers came from.

### What the code is about to do

Run all three phases against our own corpus, treating it as if it were an unlabelled client
delivery: seed and generate candidates, filter out the ones that are not worth scoring,
stratify to a workable size, and freeze a slice no tuning run may see.


In [ ]:
viz.hld([
    dict(name="Seed", tone="index", nodes=[
        ("Sample documents", "stratified by source, age and format"),
        ("Generate Q/A pairs", "with the evidence span attached"),
        ("Force multi-hop", "pair documents that share an entity"),
        ("Inject nulls", "and near-miss distractor questions")]),
    dict(name="Filter", tone="warn", nodes=[
        ("Drop self-answerable", "answerable from the question text alone"),
        ("Drop unsupported", "evidence span does not entail the answer"),
        ("Human review", "10–20% sample — the step that makes it defensible")]),
    dict(name="Maintain", tone="control", nodes=[
        ("Version with the corpus", "the snapshot it was drawn from"),
        ("Feed production failures", "every one that got a human verdict"),
        ("Hold out a frozen slice", "no tuning run is allowed to see it")]),
], title="Manufacturing an evaluation set from a client corpus",
   kicker="HLD",
   caption="A generated eval set inherits its generator's blind spots. The held-out slice and "
           "the production-failure feed are what stop it becoming a mirror of your retriever.",
   source="Deck slide 20")

In [ ]:
import random
from raglab.embed import tokenize

rng = random.Random(11)

# ── SEED ─────────────────────────────────────────────────────────────────────
# Stratify by source, not uniformly. A uniform sample of this corpus is 42%
# market commentary, and an eval set made of commentary measures nothing.
by_source = {}
for d in bundle.documents:
    by_source.setdefault(d.source, []).append(d)
seed_docs = []
for src, docs in sorted(by_source.items()):
    take = max(3, int(0.10 * len(docs)))
    seed_docs += rng.sample(docs, min(take, len(docs)))

print("SEED")
print(f"  sampled {len(seed_docs)} of {len(bundle.documents)} documents, stratified by source")
for src, docs in sorted(by_source.items()):
    n = sum(1 for d in seed_docs if d.source == src)
    print(f"    {src:<12} {n:>3} of {len(docs):>3}   ({n/len(seed_docs):.0%} of the sample)")

In [ ]:
# ── GENERATE ─────────────────────────────────────────────────────────────────
# Offline, a candidate question is generated from a passage plus its metadata. With a model
# available this is where you would prompt for (question, answer, evidence_span); the shape
# of what comes back is identical, and so is everything downstream.
candidates = []
for d in seed_docs:
    for p in d.passages:
        if not p.anchor or len(p.text.split()) < 14:
            continue
        subject = d.entities[0] if d.entities else d.title
        candidates.append({
            "query": f"According to {d.source} coverage from {d.published}, what does the "
                     f"record state about {subject}?",
            "answer": p.text.split(".")[0],
            "evidence": p.anchor,
            "doc_id": d.doc_id,
            "source": d.source,
            "published": d.published,
            "hops": 1,
        })

# Force multi-hop: pair documents that share an entity.
by_entity = {}
for d in seed_docs:
    for e in d.entities:
        by_entity.setdefault(e, []).append(d)
multi = 0
for ent, docs in by_entity.items():
    if len(docs) < 2:
        continue
    a, b = docs[0], docs[1]
    if a.doc_id == b.doc_id:
        continue
    candidates.append({
        "query": f"Combining the {a.source} and {b.source} records, what is known about {ent}?",
        "answer": f"{a.passages[0].text.split('.')[0]}; {b.passages[0].text.split('.')[0]}",
        "evidence": [a.passages[0].anchor, b.passages[0].anchor],
        "doc_id": [a.doc_id, b.doc_id], "source": "multi",
        "published": max(a.published, b.published), "hops": 2,
    })
    multi += 1

print(f"GENERATE\n  {len(candidates)} candidate questions ({multi} forced multi-hop)")

In [ ]:
# ── FILTER ───────────────────────────────────────────────────────────────────
# Two automatic gates, then a human sample. Both gates are cheap and both catch a failure
# mode that would otherwise be invisible in the final number.
def answerable_from_question_alone(c):
    '''The answer's content words already appear in the question — nothing to retrieve.'''
    a = set(tokenize(c["answer"] if isinstance(c["answer"], str) else " ".join(c["answer"])))
    q = set(tokenize(c["query"]))
    return len(a - q) / max(1, len(a)) < 0.35

def evidence_entails_answer(c):
    '''The cited span must actually support the answer.'''
    ev = c["evidence"] if isinstance(c["evidence"], list) else [c["evidence"]]
    ans = set(tokenize(c["answer"] if isinstance(c["answer"], str) else " ".join(c["answer"])))
    span = set()
    for e in ev:
        span |= set(tokenize(e))
    return len(ans & span) / max(1, len(ans)) >= 0.30

# Two deliberately flawed candidates, so you can watch the gates fire rather than trust
# that they would. Both are shapes a generator really produces.
flawed = [
    {"query": "Did Northwind Systems complete its acquisition of Tessera Analytics on "
              "2023-08-14?",
     "answer": "Northwind Systems completed its acquisition of Tessera Analytics on "
               "2023-08-14",
     "evidence": "Northwind Systems completed its acquisition of Tessera Analytics",
     "doc_id": "nw-8800", "source": "newswire", "published": "2023-08-14", "hops": 1,
     "_flaw": "answer is restated inside the question — retrieval is not being tested"},
    {"query": "What did the competition authority require of Northwind Systems?",
     "answer": "A data-portability undertaking covering customer exports",
     "evidence": "Analysts covering cloud infrastructure described the price as full but "
                 "defensible",
     "doc_id": "fl-2400", "source": "filings", "published": "2023-10-03", "hops": 1,
     "_flaw": "the cited span does not support the answer — the label is wrong"},
]
candidates += flawed

kept, dropped = [], {"self-answerable": 0, "evidence does not entail": 0}
for c in candidates:
    if answerable_from_question_alone(c):
        dropped["self-answerable"] += 1
    elif not evidence_entails_answer(c):
        dropped["evidence does not entail"] += 1
    else:
        kept.append(c)

review_n = max(1, int(0.15 * len(kept)))
print("FILTER")
for c in flawed:
    gate = ("self-answerable" if answerable_from_question_alone(c)
            else "evidence does not entail" if not evidence_entails_answer(c) else "PASSED")
    print(f"  planted flaw → caught by: {gate:<26} ({c['_flaw']})")
print()
for reason, n in dropped.items():
    print(f"  dropped: {reason:<26} {n:>4}  ({n/len(candidates):.0%} of candidates)")
print(f"  kept                              {len(kept):>4}")
print(f"  queued for human review           {review_n:>4}  (15% sample — not optional)")

In [ ]:
# ── MAINTAIN ─────────────────────────────────────────────────────────────────
by_type = {"1-hop": [c for c in kept if c["hops"] == 1],
           "2-hop": [c for c in kept if c["hops"] == 2]}
target = {"1-hop": 60, "2-hop": 40}
final = []
for t, pool in by_type.items():
    rng.shuffle(pool)
    final += pool[: target[t]]
rng.shuffle(final)
frozen = set(range(int(0.15 * len(final))))

tables.show(pd.DataFrame([
    ["Version with the corpus snapshot",
     f"corpus hash {hash(tuple(d.content_hash for d in bundle.documents)) & 0xFFFFFF:06x}",
     "A metric without a corpus version is not reproducible. Store them together or the "
     "number means nothing in three months."],
    ["Add production failures with a human verdict", "0 today — the loop starts on day one of "
     "production",
     "This is what stops the set becoming a mirror of your own retriever's blind spots."],
    ["Hold out a frozen slice", f"{len(frozen)} of {len(final)} questions (15%)",
     "You may look at it once, at the end. Every other slice has been seen by a tuning run "
     "and can be overfitted."],
    ["Keep the nulls", f"{sum(1 for x in bundle.questions if x.question_type=='null')} in the "
     "shipped set",
     "Abstention is scored. Answering a null question is a failure, not a neutral outcome."],
], columns=["Maintenance rule", "State on this run", "Why it is not optional"]),
    title="MAINTAIN — what keeps the set honest after week one",
    kicker="Eval-set lifecycle",
    caption=f"Manufactured set: {len(final)} questions from {len(seed_docs)} sampled documents, "
            f"{len(frozen)} frozen. The pipeline is the deliverable; the numbers are its output.",
    emphasize="Maintenance rule")

In [ ]:
tables.callout(
    "The set you just generated would score <i>your own retriever</i> generously, because the "
    "same assumptions produced both. Its questions use the documents' own vocabulary; a real "
    "user paraphrases. Two defences, and you need both: the <b>frozen slice</b>, which no "
    "tuning run may see, and the <b>production-failure feed</b>, which brings in the "
    "questions your generator would never have thought to write."
    "<br><br>The corpus these notebooks use was built with the opposite bias on purpose — "
    "paraphrased questions, descriptor references, and a deliberate lexical gap — which is "
    "why its numbers are lower and more useful than a set generated from the documents.",
    kind="warn", title="A generated eval set inherits its generator's blind spots")

---

## 2.4 The interview


In [ ]:
tables.show(pd.DataFrame([
    ["How would you build an evaluation dataset for a proprietary knowledge base?",
     "Whether you have actually done it, or only read about it",
     "Stratified seed → generate with evidence spans attached → force multi-hop by pairing on "
     "shared entities → inject nulls → two automatic filters → human review of a 10–20% "
     "sample → version with the corpus → freeze a slice → feed production failures back."],
    ["Which retrieval metrics would you use for a multi-hop system?",
     "Whether you know average evidence recall flatters a multi-hop system",
     "Evidence Recall@k for coverage, full-chain recall for the number that predicts a correct "
     "answer, nDCG when relevance is graded. Report all three sliced by question type."],
    ["How many questions do you need?",
     "Whether you think about the noise band before the sample size",
     "Enough that the delta you care about clears the interval. Measure it: a paired bootstrap "
     "on your current set tells you the smallest difference you can currently detect. "
     "Notebook 06 runs one."],
    ["The client says their corpus changes weekly. What happens to the eval set?",
     "Whether you version data and labels together",
     "The set is pinned to a corpus snapshot. On refresh, re-resolve every evidence span "
     "against the new chunking, report the spans that no longer resolve, and treat that count "
     "as a first-class metric — it is label rot, and it is invisible otherwise."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Typical interview questions: the eval set",
    kicker="Section 2 · interview", emphasize="Question")

---

## What carries forward

- One record, three evaluations. `evidence_list` is the field that makes retrieval
  measurable; without it you are scoring luck.
- Question type predicts the failure and names the lever — and both of those are testable
  claims, not slogans. Per-entity packing quotas and date pre-filters are levers you have now
  measured.
- Null questions cost nothing and expose the failure no other metric will show you.
- A manufactured eval set is a pipeline with a maintenance contract, not a file. The frozen
  slice and the production-failure feed are what keep it honest.

**Next:** `03_rag_system_design.ipynb` — the reference architecture, chunking as a measured
decision, keeping an index fresh without a nightly rebuild, and permission-aware retrieval.
